# NB6 (Colab GPU) — every PVSetFormer-family test row on one GPU

D80 and its extension, after the protocol freeze (frozen configuration `2d27ea1e82060a0f…`). Rows, in this order: main, A2, A4c, A6e, A6d (TimeXer), the test bed's main. All of them run on the same GPU, so the ablations are compared with the main row on one device. Same model, inputs, settings and seeds as the CPU development runs; only the device differs.

First two validation rounds on the 10 validation units with the 5 frozen seeds (150 trainings each): V12 (the main model's setting) and TX1 (TimeXer, A6d's setting). Each gate: the GPU ensemble level with the CPU one (under 3% at every horizon) and the per-seed medians within 3 standard errors. If V12 fails, the notebook stops (every row stays on the CPU). If only TimeXer fails, A6d is skipped here (it then runs on the laptop's CPU). The gates print the projected GPU hours of every row.

**How to run:** Runtime > Change runtime type > a GPU. Then Runtime > Run all. Keep the tab open. If Colab disconnects, reconnect and Run all again: finished trainings are kept on Drive and skipped. The last cell unmounts Drive. Then, on the laptop: `python tools/fetch_drive_runs.py --set nb6gpu`.

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_REPO = Path('/content/drive/MyDrive/pv-cold-start-nb6gpu')
ROOT = Path('/content/repo')
man = json.loads((DRIVE_REPO / 'MIRROR_MANIFEST.json').read_text())
if ROOT.exists():
    shutil.rmtree(ROOT)          # links are removed, not followed: Drive keeps the outputs
for f, h in man['sha256'].items():
    if f.startswith('notebooks/'):
        continue                 # this open notebook: Colab saves it to Drive itself; nothing runs from it
    d = ROOT / f
    d.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DRIVE_REPO / f, d)
    assert hashlib.sha256(d.read_bytes()).hexdigest() == h, f'{f} differs from the mirror manifest'
shutil.copyfile(DRIVE_REPO / 'MIRROR_MANIFEST.json', ROOT / 'MIRROR_MANIFEST.json')
for rel in ['results/nb6_forecasts/parts', 'results/nb6_gpu_val']:   # outputs live on Drive
    (DRIVE_REPO / rel).mkdir(parents=True, exist_ok=True)
    (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
    os.symlink(DRIVE_REPO / rel, ROOT / rel)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pvlib==0.16.1'], check=True)   # the gate's priors
import torch, numpy, pandas
assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > a GPU'
ENV = {**os.environ, 'PVSF_DEVICE': 'cuda', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'PYTHONUNBUFFERED': '1'}
PROCS = max(1, min(4, os.cpu_count()))   # trainings at once on the GPU (one CPU each)
print('mirror commit', man['commit'], '| files', len(man['sha256']), '| GPU', torch.cuda.get_device_name(0),
      '| CPUs', os.cpu_count(), '| processes', PROCS, '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| pandas', pandas.__version__)

def step(*args, check=True):
    t = time.time()
    p = subprocess.Popen([sys.executable, '-u', 'tools/nb6_gpu.py', *args], cwd=ROOT, env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    assert rc == 0 or not check, f'{args} failed (see above)'
    print(f'--- {" ".join(args)} done in {(time.time() - t) / 60:.1f} min (exit {rc})', flush=True)
    return rc

def gate_passed(ref):
    g = ROOT / 'results' / 'nb6_gpu_val' / ref / 'GATE.json'
    return g.exists() and json.loads(g.read_text())['passed']

Validation rounds (150 trainings each), then the gates. V12's gate stops the notebook if the GPU is not level with the CPU; TimeXer's gate decides only where A6d runs.

In [ ]:
step('--validate', '--ref', 'V12_5s', '--procs', str(PROCS))
step('--validate', '--ref', 'TX1_5s', '--procs', str(PROCS))
step('--gate', '--ref', 'V12_5s')                  # stops here if it fails
step('--gate', '--ref', 'TX1_5s', check=False)
TX1_OK = gate_passed('TX1_5s')
print('V12 gate passed:', gate_passed('V12_5s'), '| TimeXer gate passed:', TX1_OK)

Projected GPU time of every row, from the validation trainings' GPU run times (the gates' projection).

In [ ]:
import pandas as pd
G = [ROOT / 'results' / 'nb6_gpu_val' / r / 'GATE.json' for r in ['V12_5s', 'TX1_5s']]
P = pd.concat([pd.DataFrame(json.loads(g.read_text())['projection']) for g in G if g.exists()], ignore_index=True)
rows_here = ['main', 'A2', 'A4c', 'A6e'] + (['A6d'] if TX1_OK else []) + ['tb_main']
P = P.set_index('row').loc[rows_here]
print(P[['trainings', 'gpu_training_hours', 'processes', 'wall_hours', 'longest_training_h']].to_string())
print(f'expected GPU wall time for these rows: {P.wall_hours.sum():.1f} h')

The test rows, longest trainings first, in the author's order. Each row prints its mean training time.

In [ ]:
for row in ['main', 'A2', 'A4c', 'A6e', 'A6d', 'tb_main']:
    if row == 'A6d' and not TX1_OK:
        print('A6d: the TimeXer gate did not pass, so A6d runs on the laptop CPU (not here)')
        continue
    step('--row', row, '--procs', str(PROCS))

In [ ]:
drive.flush_and_unmount()
print('done: on the laptop run python tools/fetch_drive_runs.py --set nb6gpu')